In [1]:
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import KFold
import pandas as pd
import numpy as np
from functions import *
from sktime.datatypes._panel._convert import from_2d_array_to_nested

crossvalidation = KFold(n_splits=5, shuffle=True, random_state=42)

# features_train = pd.read_csv('../out/features_train.csv')
# features_train = features_train.fillna(features_train.median())
X_train = pd.read_csv('../data/X_train.csv', index_col='id')
X_train_filtered = filter_signal(X_train)
y_train = pd.read_csv('../data/y_train.csv', index_col='id')

# would probably make sense to start from the first R-peak to have similar starting point

In [2]:
from tslearn.utils import from_sktime_dataset, to_sktime_dataset, to_pyts_dataset
X_train_filtered_sk = from_2d_array_to_nested(X_train_filtered.dropna(axis=1))
X_train_filtered_sk.columns = ["dim_0"]
X_train_filtered_ts = from_sktime_dataset(X_train_filtered_sk)
X_train_filtered_pyts = to_pyts_dataset(X_train_filtered_ts)
X_train_filtered_sk = to_sktime_dataset(X_train_filtered_ts)

In [7]:
from pyts.classification import TimeSeriesForest

# test = to_pyts_dataset(from_sktime_dataset(X_train_filtered_ts))

time_forest = TimeSeriesForest(n_estimators = 1000, n_windows=20, random_state=42)
time_forest_baseline = np.mean(cross_val_score(time_forest, X_train_filtered_pyts, np.ravel(y_train), scoring="f1_micro", cv=crossvalidation))
print("A baseline time forest model achieves an F1 score of: ", time_forest_baseline)

A baseline time forest model achieves an F1 score of:  0.5964431359970674


In [3]:
from tslearn.svm import TimeSeriesSVC

clf = TimeSeriesSVC(kernel="gak", gamma="auto", probability=True)
clf.fit(X_train_filtered_ts, np.ravel(y_train))

KeyboardInterrupt: 

In [ ]:
from tslearn.shapelets import LearningShapelets

clf = LearningShapelets(n_shapelets_per_size={4: 5}, max_iter=1, verbose=0)
clf.fit(X_train_filtered_ts, np.ravel(y_train))

ModuleNotFoundError: No module named 'tensorflow'

In [39]:
from pyts.classification import SAXVSM

clf = SAXVSM(window_size=34, sublinear_tf=False, use_idf=False)
clf.fit(X_train_filtered_pyts, np.ravel(y_train))


SAXVSM(sublinear_tf=False, use_idf=False, window_size=34)

In [ ]:
from pyts.classification import BOSSVS

clf = BOSSVS(window_size=28)
clf.fit(X_train_filtered_pyts, np.ravel(y_train))


MemoryError: 

# Those somehow dont work

In [25]:
from sktime.classification.hybrid import HIVECOTEV1
from sktime.contrib.vector_classifiers._rotation_forest import RotationForest
clf = HIVECOTEV1(
    stc_params={
        "estimator": RotationForest(n_estimators=3),
        "n_shapelet_samples": 500,
        "max_shapelets": 20,
        "batch_size": 100,
    },
    tsf_params={"n_estimators": 10},
    rise_params={"n_estimators": 10},
    cboss_params={"n_parameter_samples": 25, "max_ensemble_size": 5},
)
clf.fit(X_train_filtered_sk.values.tolist(), pd.Series(y_train))

ValueError: The truth value of a DataFrame is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().

In [11]:
from sktime.classification.hybrid import HIVECOTEV2
hc2 = HIVECOTEV2()
hc2.fit(X_train_filtered_sk, pd.Series(y_train))

ValueError: The truth value of a DataFrame is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().

In [ ]:
np.random.normal(0, 0.1, 1000)